This notebook is to show the initial filtering of candidates inorder to reduce the dataset size from 100k to close to top 5k, using only deterministic methods

In [1]:
import pandas as pd
import json

df = pd.read_json(
    "/Users/sachisingh/Downloads/so/India_runs_data_and_ai_challenge/candidates.json",
    lines=True
)

print(df.columns.tolist())
print(df.iloc[0].to_dict().keys())

['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals']
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [4]:
len(df)

100000

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# Honeypot Removal

the honeypot removal is based on metadata , redrobsignals ,impossible figures and inconsitancy if timelines

In [2]:
df["redrob_signals"].head()

0    {'profile_completeness_score': 86.9, 'signup_d...
1    {'profile_completeness_score': 78.7, 'signup_d...
2    {'profile_completeness_score': 31.9, 'signup_d...
3    {'profile_completeness_score': 28.5, 'signup_d...
4    {'profile_completeness_score': 84.6, 'signup_d...
Name: redrob_signals, dtype: object

In [32]:
nested_cols = ["redrob_signals", "profile"]

flat_parts = [
    pd.json_normalize(df[col]).add_prefix(col + ".")
    for col in nested_cols
]

base = df.drop(columns=nested_cols, errors="ignore")

candidate_df = pd.concat([base] + flat_parts, axis=1)

In [33]:
import pandas as pd

# ---------------------------------
# Convert dates
# ---------------------------------

candidate_df["redrob_signals.signup_date"] = pd.to_datetime(
    candidate_df["redrob_signals.signup_date"],
    errors="coerce"
)

candidate_df["redrob_signals.last_active_date"] = pd.to_datetime(
    candidate_df["redrob_signals.last_active_date"],
    errors="coerce"
)

# ---------------------------------
# Initialize
# ---------------------------------

candidate_df["honeypot_score"] = 0
candidate_df["honeypot_reasons"] = ""

# ---------------------------------
# Rule 1: Salary min > max
# ---------------------------------

rule_salary = (
    candidate_df["redrob_signals.expected_salary_range_inr_lpa.min"]
    >
    candidate_df["redrob_signals.expected_salary_range_inr_lpa.max"]
)

candidate_df.loc[rule_salary, "honeypot_score"] += 50
candidate_df.loc[
    rule_salary,
    "honeypot_reasons"
] += "|INVALID_SALARY"

# ---------------------------------
# Rule 2: Last active before signup
# ---------------------------------

rule_dates = (
    candidate_df["redrob_signals.last_active_date"]
    <
    candidate_df["redrob_signals.signup_date"]
)

candidate_df.loc[rule_dates, "honeypot_score"] += 50
candidate_df.loc[
    rule_dates,
    "honeypot_reasons"
] += "|LAST_ACTIVE_BEFORE_SIGNUP"

# ---------------------------------
# Rule 3: Recruiter response rate
# should be between 0 and 1
# ---------------------------------

rule_response = (
    (candidate_df["redrob_signals.recruiter_response_rate"] < 0)
    |
    (candidate_df["redrob_signals.recruiter_response_rate"] > 1)
)

candidate_df.loc[rule_response, "honeypot_score"] += 20
candidate_df.loc[
    rule_response,
    "honeypot_reasons"
] += "|INVALID_RESPONSE_RATE"

# ---------------------------------
# Rule 4: Interview completion rate
# should be between 0 and 1
# ---------------------------------

rule_interview = (
    (candidate_df["redrob_signals.interview_completion_rate"] < 0)
    |
    (candidate_df["redrob_signals.interview_completion_rate"] > 1)
)

candidate_df.loc[rule_interview, "honeypot_score"] += 20
candidate_df.loc[
    rule_interview,
    "honeypot_reasons"
] += "|INVALID_INTERVIEW_RATE"

# ---------------------------------
# Rule 5: Profile completeness
# should be between 0 and 100
# ---------------------------------

rule_profile = (
    (candidate_df["redrob_signals.profile_completeness_score"] < 0)
    |
    (candidate_df["redrob_signals.profile_completeness_score"] > 100)
)

candidate_df.loc[rule_profile, "honeypot_score"] += 20
candidate_df.loc[
    rule_profile,
    "honeypot_reasons"
] += "|INVALID_PROFILE_COMPLETENESS"

# ---------------------------------
# Rule 6: Negative response time
# ---------------------------------

rule_response_time = (
    candidate_df["redrob_signals.avg_response_time_hours"] < 0
)

candidate_df.loc[
    rule_response_time,
    "honeypot_score"
] += 20

candidate_df.loc[
    rule_response_time,
    "honeypot_reasons"
] += "|NEGATIVE_RESPONSE_TIME"

# ---------------------------------
# DEFINITELY FAKE
# only remove candidates with
# multiple hard violations
# ---------------------------------

candidate_df["is_honeypot"] = (
    candidate_df["honeypot_score"] >= 50
)

print(f"Total candidates: {len(candidate_df):,}")
print(f"Honeypots: {candidate_df['is_honeypot'].sum():,}")
print(f"Remaining: {(~candidate_df['is_honeypot']).sum():,}")

# ---------------------------------
# Breakdown
# ---------------------------------

print("\nRule counts")
print("Invalid salary:", rule_salary.sum())
print("Bad dates:", rule_dates.sum())
print("Bad response:", rule_response.sum())
print("Bad interview:", rule_interview.sum())
print("Bad profile:", rule_profile.sum())
print("Bad response time:", rule_response_time.sum())

# ---------------------------------
# Inspect
# ---------------------------------

display(
    candidate_df.loc[
        candidate_df["is_honeypot"],
        [
            "candidate_id",
            "honeypot_score",
            "honeypot_reasons",
            "redrob_signals.expected_salary_range_inr_lpa.min",
            "redrob_signals.expected_salary_range_inr_lpa.max",
            "redrob_signals.signup_date",
            "redrob_signals.last_active_date"
        ]
    ]
    .sort_values("honeypot_score", ascending=False)
    .head(50)
)

# ---------------------------------
# Clean pool
# ---------------------------------

clean_candidates = candidate_df.loc[
    ~candidate_df["is_honeypot"]
].copy()

print(f"\nClean pool: {len(clean_candidates):,}")

Total candidates: 100,000
Honeypots: 24,895
Remaining: 75,105

Rule counts
Invalid salary: 18865
Bad dates: 7496
Bad response: 0
Bad interview: 0
Bad profile: 0
Bad response time: 0


,candidate_id,honeypot_score,honeypot_reasons,redrob_signals.expected_salary_range_inr_lpa.min,redrob_signals.expected_salary_range_inr_lpa.max,redrob_signals.signup_date,redrob_signals.last_active_date
79695,CAND_0079696,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,16.4,7.2,2026-01-31,2025-10-31
57694,CAND_0057695,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,13.8,9.5,2026-04-05,2025-12-06
57837,CAND_0057838,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,17.5,11.4,2025-10-25,2025-10-08
57830,CAND_0057831,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,12.2,8.3,2026-03-13,2026-01-10
92072,CAND_0092073,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,17.9,6.7,2026-03-14,2026-01-20
20393,CAND_0020394,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,10.4,6.8,2026-01-24,2025-12-23
8026,CAND_0008027,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,15.7,8.0,2026-05-12,2026-05-11
43533,CAND_0043534,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,17.0,6.5,2026-03-02,2025-10-24
80005,CAND_0080006,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,16.1,8.0,2026-05-05,2026-01-09
79985,CAND_0079986,100,|INVALID_SALARY|LAST_ACTIVE_BEFORE_SIGNUP,13.1,12.7,2026-05-05,2025-11-28



Clean pool: 75,105


In [11]:
print("Invalid salary:", rule_salary.sum())
print("Bad dates:", rule_dates.sum())
print("Bad response:", rule_response.sum())
print("Bad interview:", rule_interview.sum())

print("Bad profile:", rule_profile.sum())
print("Bad response time:", rule_response_time.sum())

Invalid salary: 18865
Bad dates: 7496
Bad response: 0
Bad interview: 0
Bad profile: 0
Bad response time: 0


## education + career timeline inconsistencies.

In [13]:
print(candidate_df["education"].iloc[0])
print()
print(candidate_df["career_history"].iloc[0])

[{'institution': 'Lovely Professional University', 'degree': 'B.E.', 'field_of_study': 'Computer Science', 'start_year': 2017, 'end_year': 2020, 'grade': '8.24 CGPA', 'tier': 'tier_3'}]

[{'company': 'Mindtree', 'title': 'Backend Engineer', 'start_date': '2024-03-08', 'end_date': None, 'duration_months': 27, 'is_current': True, 'industry': 'IT Services', 'company_size': '10001+', 'description': 'Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.'}, {'company': 'Dunder Mifflin', 'title': 'Analytics Engineer', 'start_date': '2019-07-03', 'end_date': '2024-01-08', 'duration_months': 55, 'is_current': Fal

In [34]:
import pandas as pd

def invalid_education_timeline(education):

    if not isinstance(education, list):
        return False

    for edu in education:

        start = edu.get("start_year")
        end = edu.get("end_year")

        if start and end and start > end:
            return True

    return False


def invalid_job_timeline(career_history):

    if not isinstance(career_history, list):
        return False

    for job in career_history:

        start = job.get("start_date")
        end = job.get("end_date")

        if start and end:

            if pd.to_datetime(start) > pd.to_datetime(end):
                return True

    return False


def duration_mismatch(career_history):

    if not isinstance(career_history, list):
        return False

    for job in career_history:

        start = job.get("start_date")
        end = job.get("end_date")
        reported = job.get("duration_months")

        if start is None or reported is None:
            continue

        start = pd.to_datetime(start)

        if end is None:
            end = pd.Timestamp.today()
        else:
            end = pd.to_datetime(end)

        actual_months = (end - start).days / 30.44

        if abs(actual_months - reported) > 6:
            return True

    return False

In [35]:
# =====================================================
# RUN CHECKS ON CLEAN CANDIDATES
# =====================================================

clean_candidates["bad_education"] = (
    clean_candidates["education"]
    .apply(invalid_education_timeline)
)

clean_candidates["bad_job_dates"] = (
    clean_candidates["career_history"]
    .apply(invalid_job_timeline)
)

clean_candidates["bad_duration"] = (
    clean_candidates["career_history"]
    .apply(duration_mismatch)
)

# =====================================================
# SCORE
# =====================================================

clean_candidates["timeline_honeypot_score"] = 0

clean_candidates.loc[
    clean_candidates["bad_education"],
    "timeline_honeypot_score"
] += 40

clean_candidates.loc[
    clean_candidates["bad_job_dates"],
    "timeline_honeypot_score"
] += 40

clean_candidates.loc[
    clean_candidates["bad_duration"],
    "timeline_honeypot_score"
] += 30

# Only keep genuinely impossible profiles
clean_candidates["timeline_honeypot"] = (
    clean_candidates["timeline_honeypot_score"] >= 30
)

# =====================================================
# SUMMARY
# =====================================================

print("\n=== TIMELINE CHECKS ===")

print(
    "Bad education:",
    clean_candidates["bad_education"].sum()
)

print(
    "Bad job dates:",
    clean_candidates["bad_job_dates"].sum()
)

print(
    "Bad duration:",
    clean_candidates["bad_duration"].sum()
)

print(
    "\nTimeline honeypots:",
    clean_candidates["timeline_honeypot"].sum()
)

# =====================================================
# INSPECT
# =====================================================

display(
    clean_candidates.loc[
        clean_candidates["timeline_honeypot"],
        [
            "candidate_id",
            "timeline_honeypot_score",
            "bad_education",
            "bad_job_dates",
            "bad_duration"
        ]
    ]
    .sort_values(
        "timeline_honeypot_score",
        ascending=False
    )
    .head(50)
)

# =====================================================
# FINAL CLEAN POOL
# =====================================================

final_clean_candidates = clean_candidates.loc[
    ~clean_candidates["timeline_honeypot"]
].copy()

print(
    f"\nFinal clean candidates: {len(final_clean_candidates):,}"
)


=== TIMELINE CHECKS ===
Bad education: 0
Bad job dates: 0
Bad duration: 24

Timeline honeypots: 24


,candidate_id,timeline_honeypot_score,bad_education,bad_job_dates,bad_duration
5290,CAND_0005291,30,False,False,True
7412,CAND_0007413,30,False,False,True
91067,CAND_0091068,30,False,False,True
90899,CAND_0090900,30,False,False,True
74118,CAND_0074119,30,False,False,True
66404,CAND_0066405,30,False,False,True
64076,CAND_0064077,30,False,False,True
57710,CAND_0057711,30,False,False,True
55684,CAND_0055685,30,False,False,True
53733,CAND_0053734,30,False,False,True



Final clean candidates: 75,081


# NON-MATCHES

In [ ]:
remove obvious non-target candidates

In [24]:
final_clean_candidates.head()

,candidate_id,profile,career_history,education,skills,certifications,languages,redrob_signals.profile_completeness_score,redrob_signals.signup_date,redrob_signals.last_active_date,...,redrob_signals.skill_assessment_scores.Deep Learning,redrob_signals.skill_assessment_scores.QLoRA,honeypot_score,honeypot_reasons,is_honeypot,bad_education,bad_job_dates,bad_duration,timeline_honeypot_score,timeline_honeypot
0,CAND_0000001,"{'anonymized_name': 'Ira Vora', 'headline': 'B...","[{'company': 'Mindtree', 'title': 'Backend Eng...",[{'institution': 'Lovely Professional Universi...,"[{'name': 'Tailwind', 'proficiency': 'intermed...",[],"[{'language': 'English', 'proficiency': 'profe...",86.9,2025-10-16,2026-05-20,...,NaN,NaN,0,,False,False,False,False,0,False
1,CAND_0000002,"{'anonymized_name': 'Saanvi Sethi', 'headline'...","[{'company': 'Wipro', 'title': 'Operations Man...","[{'institution': 'Local Engineering College', ...","[{'name': 'Project Management', 'proficiency':...",[],"[{'language': 'English', 'proficiency': 'profe...",78.7,2025-07-28,2025-11-12,...,NaN,NaN,0,,False,False,False,False,0,False
2,CAND_0000003,"{'anonymized_name': 'Yash Agarwal', 'headline'...","[{'company': 'TCS', 'title': 'Customer Support...","[{'institution': 'Local Engineering College', ...","[{'name': 'Angular', 'proficiency': 'intermedi...",[],"[{'language': 'English', 'proficiency': 'profe...",31.9,2024-08-02,2026-03-21,...,NaN,NaN,0,,False,False,False,False,0,False
3,CAND_0000004,"{'anonymized_name': 'Anil Bose', 'headline': '...","[{'company': 'Dunder Mifflin', 'title': 'Marke...","[{'institution': 'Local Engineering College', ...","[{'name': 'Node.js', 'proficiency': 'intermedi...","[{'name': 'AWS Certified Cloud Practitioner', ...","[{'language': 'English', 'proficiency': 'profe...",28.5,2025-07-21,2026-03-25,...,NaN,NaN,0,,False,False,False,False,0,False
4,CAND_0000005,"{'anonymized_name': 'Aisha Sethi', 'headline':...","[{'company': 'Stark Industries', 'title': 'Acc...","[{'institution': 'Chandigarh University', 'deg...","[{'name': 'SQL', 'proficiency': 'beginner', 'e...",[],"[{'language': 'English', 'proficiency': 'profe...",84.6,2023-10-07,2025-10-01,...,NaN,NaN,0,,False,False,False,False,0,False


In [31]:
final_clean_candidates.columns

Index(['candidate_id', 'profile', 'career_history', 'education', 'skills',
       'certifications', 'languages',
       'redrob_signals.profile_completeness_score',
       'redrob_signals.signup_date', 'redrob_signals.last_active_date',
       'redrob_signals.open_to_work_flag',
       'redrob_signals.profile_views_received_30d',
       'redrob_signals.applications_submitted_30d',
       'redrob_signals.recruiter_response_rate',
       'redrob_signals.avg_response_time_hours',
       'redrob_signals.connection_count',
       'redrob_signals.endorsements_received',
       'redrob_signals.notice_period_days',
       'redrob_signals.preferred_work_mode',
       'redrob_signals.willing_to_relocate',
       'redrob_signals.github_activity_score',
       'redrob_signals.search_appearance_30d',
       'redrob_signals.saved_by_recruiters_30d',
       'redrob_signals.interview_completion_rate',
       'redrob_signals.offer_acceptance_rate', 'redrob_signals.verified_email',
       'redrob_signal

In [36]:
import pandas as pd

# =====================================================
# OBVIOUS NON-TARGET CURRENT ROLES
# =====================================================

REJECT_TERMS = [
    "recruiter",
    "talent acquisition",
    "human resources",
    "hr manager",
    "customer support",
    "technical support",
    "sales representative",
    "sales executive",
    "business development",
    "marketing manager",
    "marketing executive",
    "accountant",
    "finance manager",
    "graphic designer",
    "ui designer",
    "ux designer",
    "teacher",
    "professor",
    "attorney",
    "lawyer"
]

# =====================================================
# BUILD CURRENT ROLE TEXT ONLY
# =====================================================

final_clean_candidates["current_role_text"] = (
    final_clean_candidates["profile.current_title"]
        .fillna("")
        .astype(str)
        .str.lower()
    +
    " " +
    final_clean_candidates["profile.headline"]
        .fillna("")
        .astype(str)
        .str.lower()
)

# =====================================================
# REJECT CHECK
# =====================================================

def obvious_reject(text):

    for term in REJECT_TERMS:
        if term in text:
            return True

    return False

final_clean_candidates["obvious_reject"] = (
    final_clean_candidates["current_role_text"]
    .apply(obvious_reject)
)

# =====================================================
# INSPECT BEFORE DROPPING
# =====================================================

print(
    "Obvious rejects found:",
    final_clean_candidates["obvious_reject"].sum()
)

display(
    final_clean_candidates.loc[
        final_clean_candidates["obvious_reject"],
        [
            "candidate_id",
            "profile.current_title",
            "profile.headline",
            "profile.years_of_experience"
        ]
    ]
    .sample(
        min(
            50,
            final_clean_candidates["obvious_reject"].sum()
        )
    )
)

# =====================================================
# REMOVE OBVIOUS REJECTS
# =====================================================

filtered_candidates = (
    final_clean_candidates.loc[
        ~final_clean_candidates["obvious_reject"]
    ]
    .copy()
)

print(
    "\nBefore:",
    len(final_clean_candidates)
)

print(
    "After:",
    len(filtered_candidates)
)

print(
    "Removed:",
    len(final_clean_candidates)
    - len(filtered_candidates)
)

Obvious rejects found: 24714


,candidate_id,profile.current_title,profile.headline,profile.years_of_experience
90946,CAND_0090947,Sales Executive,Sales Executive | 9.9+ yrs experience,9.9
8189,CAND_0008190,Accountant,Accountant | Driving business outcomes,13.5
67086,CAND_0067087,Marketing Manager,Marketing Manager | Helping teams scale,12.7
77618,CAND_0077619,Graphic Designer,Graphic Designer | Driving business outcomes,10.3
27334,CAND_0027335,Customer Support,Customer Support | Driving business outcomes,3.3
18516,CAND_0018517,Sales Executive,Sales Executive | Driving business outcomes,5.4
94903,CAND_0094904,Accountant,Accountant | Helping teams scale,8.9
74430,CAND_0074431,Accountant,Accountant | Helping teams scale,5.2
73027,CAND_0073028,Sales Executive,Sales Executive | Driving business outcomes,4.0
90907,CAND_0090908,Customer Support,Customer Support | Helping teams scale,11.0



Before: 75081
After: 50367
Removed: 24714


In [37]:
final_clean_candidates.loc[
    final_clean_candidates["obvious_reject"],
    [
        "candidate_id",
        "profile.current_title",
        "profile.headline",
        "profile.summary"
    ]
].sample(100, random_state=42)

,candidate_id,profile.current_title,profile.headline,profile.summary
16638,CAND_0016639,Accountant,Accountant | 13.0+ yrs experience,Professional with 13.0+ years of experience. M...
44476,CAND_0044477,HR Manager,HR Manager | Driving business outcomes,Professional with 10.4+ years of experience. M...
28280,CAND_0028281,Sales Executive,Sales Executive | Exploring AI & GenAI applica...,Sales Executive with 2.0+ years of experience ...
6906,CAND_0006907,Sales Executive,Sales Executive | Driving business outcomes,Professional with 8.1+ years of experience. My...
97584,CAND_0097585,Marketing Manager,Marketing Manager | 9.2+ yrs experience,Professional with 9.2+ years of experience. I'...
...,...,...,...,...
59833,CAND_0059834,Customer Support,Customer Support | Helping teams scale,Professional with 10.3+ years of experience. M...
89954,CAND_0089955,Marketing Manager,Marketing Manager | 5.9+ yrs experience,Professional with 5.9+ years of experience. I'...
34299,CAND_0034300,Sales Executive,Sales Executive | Driving business outcomes,Professional with 4.4+ years of experience. I'...
1003,CAND_0001004,Accountant,Accountant | Driving business outcomes,Professional with 1.9+ years of experience. My...


In [38]:
filtered_candidates["profile.current_title"] \
    .value_counts() \
    .head(50)

profile.current_title
Business Analyst                    4253
Project Manager                     4239
Mechanical Engineer                 4175
Operations Manager                  4140
Content Writer                      4130
Civil Engineer                      4084
Software Engineer                   2793
Full Stack Developer                2320
Cloud Engineer                      2277
DevOps Engineer                     2254
.NET Developer                      2231
Java Developer                      2216
Frontend Engineer                   2203
Mobile Developer                    2171
QA Engineer                         2163
Data Engineer                        646
Analytics Engineer                   636
Data Analyst                         623
Backend Engineer                     598
Senior Data Engineer                 581
Senior Software Engineer             572
ML Engineer                          149
AI Research Engineer                 135
Data Scientist                     

# positive keywords filtering

In [39]:
import pandas as pd

# =====================================================
# POSITIVE TERMS
# =====================================================

POSITIVE_TERMS = [

    # Core AI/ML
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "llm",
    "rag",
    "nlp",
    "computer vision",
    "recommendation",
    "ranking",
    "retrieval",
    "semantic search",
    "vector search",

    # Frameworks
    "pytorch",
    "tensorflow",
    "keras",
    "scikit",
    "hugging face",
    "transformers",

    # Search / Retrieval
    "faiss",
    "elasticsearch",
    "opensearch",
    "bm25",
    "weaviate",
    "qdrant",
    "pinecone",
    "milvus",
    "pgvector",

    # Data Engineering
    "data engineering",
    "data engineer",
    "etl",
    "elt",
    "spark",
    "pyspark",
    "airflow",
    "kafka",
    "snowflake",
    "databricks",
    "bigquery",

    # MLOps
    "mlops",
    "mlflow",
    "kubeflow",
    "feature engineering",
    "model deployment",

    # Programming
    "python",
    "sql",

    # Data Science
    "data science",
    "forecasting",
    "statistical modeling",

    # Backend
    "backend",
    "distributed systems",
    "microservices",
    "api",

    # LLM Stack
    "langchain",
    "llamaindex",
    "embeddings",
    "lora",
    "qlora",
    "fine tuning"
]

# =====================================================
# TITLE BONUS
# =====================================================

TITLE_BONUS = {

    "Applied ML Engineer": 10,
    "ML Engineer": 10,
    "Machine Learning Engineer": 10,
    "AI Engineer": 10,
    "Search Engineer": 10,
    "NLP Engineer": 10,
    "Computer Vision Engineer": 10,
    "Recommendation Systems Engineer": 10,

    "Senior NLP Engineer": 9,
    "Senior Machine Learning Engineer": 9,
    "Senior AI Engineer": 9,
    "Lead AI Engineer": 9,

    "Data Scientist": 8,
    "Senior Data Scientist": 8,
    "AI Research Engineer": 8,
    "AI Specialist": 8,

    "Data Engineer": 6,
    "Senior Data Engineer": 6,
    "Analytics Engineer": 6,
    "Backend Engineer": 5,

    "Senior Software Engineer (ML)": 7
}

# =====================================================
# BUILD SEARCHABLE TEXT
# =====================================================

def build_candidate_text(row):

    text = []

    text.append(str(row.get("profile.current_title", "")))
    text.append(str(row.get("profile.headline", "")))
    text.append(str(row.get("profile.summary", "")))

    skills = row.get("skills", [])

    if isinstance(skills, list):
        for skill in skills:
            text.append(
                str(skill.get("name", ""))
            )

    jobs = row.get("career_history", [])

    if isinstance(jobs, list):
        for job in jobs:

            text.append(
                str(job.get("title", ""))
            )

            text.append(
                str(job.get("description", ""))
            )

    return " ".join(text).lower()

# =====================================================
# CREATE TEXT
# =====================================================

filtered_candidates["candidate_text"] = (
    filtered_candidates.apply(
        build_candidate_text,
        axis=1
    )
)

# =====================================================
# KEYWORD SCORE
# =====================================================

def keyword_score(text):

    return sum(
        term in text
        for term in POSITIVE_TERMS
    )

filtered_candidates["relevance_score"] = (
    filtered_candidates["candidate_text"]
    .apply(keyword_score)
)

# =====================================================
# TITLE BONUS
# =====================================================

def title_bonus(title):

    return TITLE_BONUS.get(
        str(title).strip(),
        0
    )

filtered_candidates["title_bonus"] = (
    filtered_candidates["profile.current_title"]
    .apply(title_bonus)
)

# =====================================================
# FINAL RETRIEVAL SCORE
# =====================================================

filtered_candidates["retrieval_score"] = (
    filtered_candidates["relevance_score"]
    +
    filtered_candidates["title_bonus"]
)

# =====================================================
# STRONG AI TITLES
# =====================================================

GOOD_TITLES = list(
    TITLE_BONUS.keys()
)

title_keep = (
    filtered_candidates["profile.current_title"]
    .isin(GOOD_TITLES)
)

# =====================================================
# HIGH KEYWORD SIGNAL
# =====================================================

keyword_keep = (
    filtered_candidates["relevance_score"] >= 12
)

# =====================================================
# FINAL CANDIDATES
# =====================================================

ai_candidates = (
    filtered_candidates.loc[
        title_keep | keyword_keep
    ]
    .copy()
)

# =====================================================
# SUMMARY
# =====================================================

print("Before:", len(filtered_candidates))
print("After:", len(ai_candidates))
print("Removed:", len(filtered_candidates) - len(ai_candidates))

# =====================================================
# TITLE DISTRIBUTION
# =====================================================

display(

    ai_candidates[
        "profile.current_title"
    ]
    .value_counts()
    .head(50)

)

# =====================================================
# TOP CANDIDATES
# =====================================================

display(

    ai_candidates[
        [
            "candidate_id",
            "profile.current_title",
            "relevance_score",
            "title_bonus",
            "retrieval_score"
        ]
    ]
    .sort_values(
        "retrieval_score",
        ascending=False
    )
    .head(100)

)

Before: 50367
After: 6806
Removed: 43561


profile.current_title
Data Engineer                       646
Analytics Engineer                  636
Software Engineer                   629
Data Analyst                        603
Backend Engineer                    598
Senior Data Engineer                581
Senior Software Engineer            553
Civil Engineer                      202
Project Manager                     186
Mechanical Engineer                 180
Operations Manager                  180
Business Analyst                    179
Content Writer                      169
ML Engineer                         149
AI Research Engineer                135
Data Scientist                      133
Senior Software Engineer (ML)       129
Junior ML Engineer                  120
Computer Vision Engineer            115
AI Specialist                       112
Cloud Engineer                       70
Full Stack Developer                 61
QA Engineer                          53
DevOps Engineer                      51
.NET Developer    

,candidate_id,profile.current_title,relevance_score,title_bonus,retrieval_score
20876,CAND_0020877,Applied ML Engineer,26,10,36
44882,CAND_0044883,AI Engineer,25,10,35
83306,CAND_0083307,Search Engineer,24,10,34
41593,CAND_0041594,Computer Vision Engineer,23,10,33
4971,CAND_0004972,ML Engineer,23,10,33
...,...,...,...,...,...
50875,CAND_0050876,Applied ML Engineer,20,10,30
15056,CAND_0015057,ML Engineer,20,10,30
46107,CAND_0046108,ML Engineer,20,10,30
49537,CAND_0049538,Applied ML Engineer,20,10,30


In [40]:
ai_candidates["profile.current_title"] \
    .value_counts() 

profile.current_title
Data Engineer                       646
Analytics Engineer                  636
Software Engineer                   629
Data Analyst                        603
Backend Engineer                    598
Senior Data Engineer                581
Senior Software Engineer            553
Civil Engineer                      202
Project Manager                     186
Mechanical Engineer                 180
Operations Manager                  180
Business Analyst                    179
Content Writer                      169
ML Engineer                         149
AI Research Engineer                135
Data Scientist                      133
Senior Software Engineer (ML)       129
Junior ML Engineer                  120
Computer Vision Engineer            115
AI Specialist                       112
Cloud Engineer                       70
Full Stack Developer                 61
QA Engineer                          53
DevOps Engineer                      51
.NET Developer    

In [41]:
ai_candidates.groupby(
    "profile.current_title"
)["retrieval_score"].agg(
    ["count", "mean", "median", "max"]
).sort_values(
    "mean",
    ascending=False
)

,count,mean,median,max
profile.current_title,,,,
Applied ML Engineer,18,29.277778,29.0,36
Senior NLP Engineer,6,29.166667,29.0,33
NLP Engineer,14,29.142857,28.5,33
Search Engineer,22,28.727273,29.0,34
AI Engineer,20,28.550000,28.5,35
Machine Learning Engineer,23,28.304348,29.0,31
Recommendation Systems Engineer,26,28.115385,28.0,33
Computer Vision Engineer,115,27.060870,27.0,33
ML Engineer,149,27.006711,27.0,33


In [42]:
candidate_pool = ai_candidates[
    ai_candidates["retrieval_score"] >= 15
].copy()

In [43]:
len(candidate_pool)

4667

In [44]:
candidate_pool.to_json("candidate_pool.json", orient="records", indent=2)

# JOB DESCRIPTION

In [45]:
from docx import Document

file_path = "/Users/sachisingh/Downloads/so/India_runs_data_and_ai_challenge/job_description.docx"

doc = Document(file_path)

text = "\n\n".join(
    para.text.strip()
    for para in doc.paragraphs
    if para.text.strip()
)

print(text)

Job Description: Senior AI Engineer — Founding Team

Company: Redrob AI (Series A AI-native talent intelligence platform)

Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities

Employment Type: Full-time

Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role

We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.

If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.

If you've spent your career bouncing between early-stage startups and you want to "just code" without having to

In [47]:
import re
import json
import google.generativeai as genai

# Configure Gemini
genai.configure(api_key="API_KEY")

model = genai.GenerativeModel("gemini-flash-latest")

jd_text = text

prompt = f"""
You are an elite recruiting intelligence system.

Your job is NOT to summarize the JD.

Your job is to reverse-engineer what hiring managers actually care about.

Analyze the following job description and return ONLY valid JSON.

JOB DESCRIPTION:
{jd_text}

OUTPUT SCHEMA:

{{
  "role_title": "",
  "role_category": "",

  "experience": {{
    "min_years": 0,
    "max_years": 0
  }},

  "core_problem": [],

  "must_have_skills": [],

  "nice_to_have_skills": [],

  "required_domains": [],

  "required_systems_experience": [],

  "behavioral_requirements": [],

  "hidden_requirements": [],

  "negative_signals": [],

  "disqualifiers": [],

  "preferred_candidate_archetypes": [],

  "bad_candidate_archetypes": [],

  "must_have_evidence": [],

  "location_requirements": "",

  "what_the_jd_really_means": [],

  "evidence_strength": {{
      "built_production_ranking_system": 100,
      "built_recommendation_system": 95,
      "owned_search_platform": 95,
      "production_retrieval_system": 90,
      "production_ml_system": 80,
      "relevant_job_title": 75,
      "open_source_contributions": 70,
      "published_paper": 60,
      "side_project": 40,
      "listed_skill": 20,
      "course_or_certificate": 10
  }},

  "scoring_dimensions": [
    {{
      "name": "",
      "weight": 0,
      "reason": ""
    }}
  ],

  "ideal_candidate_summary": ""
}}

IMPORTANT:

- Return ONLY JSON.
- Do not wrap in markdown.
- Infer hidden hiring intent.
- Prefer career evidence over skill keywords.
- Identify the actual archetype being hired.
"""

response = model.generate_content(prompt)

response_text = response.text.strip()

# Remove markdown wrappers if Gemini adds them
if response_text.startswith("```json"):
    response_text = response_text.replace("```json", "").replace("```", "").strip()
elif response_text.startswith("```"):
    response_text = response_text.replace("```", "").strip()

jd_structured = json.loads(response_text)

print(json.dumps(jd_structured, indent=2))

# Optional save
with open("structured_jd.json", "w") as f:
    json.dump(jd_structured, f, indent=2)

{
  "role_title": "Senior AI Engineer \u2014 Founding Team",
  "role_category": "AI / Machine Learning Engineering",
  "experience": {
    "min_years": 5,
    "max_years": 9
  },
  "core_problem": [
    "Overhauling legacy BM25/rule-based heuristics to create a modern hybrid vector search, retrieval, and matching pipeline.",
    "Establishing rigorous evaluation frameworks (offline and online) to measure and iterate on ranking system performance.",
    "Scaling and owning the production intelligence layer of a talent marketplace from scratch as a founding engineer."
  ],
  "must_have_skills": [
    "Deploying embeddings-based retrieval systems (sentence-transformers, BGE, E5, etc.) in production",
    "Operating vector databases or hybrid search infrastructure (Pinecone, OpenSearch, Milvus, Qdrant, Elasticsearch, FAISS)",
    "Production-grade Python software engineering",
    "Designing and implementing ranking evaluation metrics (NDCG, MRR, MAP, offline-to-online correlation, A/B tes

In [ ]:
import json

with open("jd_structured.json", "w") as f:
    json.dump(jd_structured, f, indent=2)